In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Palo_Alto_Networks.csv")
print(df.shape)

text_cols = df.select_dtypes(include="object").columns.tolist()
print(text_cols)

(1470, 31)
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


C:\Users\ranou\AppData\Local\Temp\ipykernel_23048\878246014.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns.tolist()


Converting text columns to numbers

In [2]:
df_enc = df.copy()

df_enc["OverTime"] = df_enc["OverTime"].map({"No": 0, "Yes": 1})
df_enc["Gender"] = df_enc["Gender"].map({"Female": 0, "Male": 1})

multi_cols = ["BusinessTravel", "Department", "EducationField", "JobRole", "MaritalStatus"]
df_enc = pd.get_dummies(df_enc, columns=multi_cols, dtype=int)

print(df_enc.shape)
print(df_enc.select_dtypes(exclude="number").columns.tolist())

(1470, 50)
[]


Split the data into training and test sets

In [3]:
# 1. Engineered features
df_enc["IncomeToExperience"] = df_enc["MonthlyIncome"] / (df_enc["TotalWorkingYears"] + 1)
df_enc["PromotionDelay"] = df_enc["YearsSinceLastPromotion"] / (df_enc["YearsAtCompany"] + 1)
df_enc["EngagementScore"] = df_enc[
    ["JobSatisfaction", "EnvironmentSatisfaction", "RelationshipSatisfaction", "JobInvolvement"]
].mean(axis=1)
df_enc["WorkloadStress"] = (
    (df_enc["OverTime"] == 1) & (df_enc["BusinessTravel_Travel_Frequently"] == 1)
).astype(int)

# 2. Train/test split
from sklearn.model_selection import train_test_split

X = df_enc.drop("Attrition", axis=1)
y = df_enc["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (1176, 53)
X_test: (294, 53)


Scaling the numeric columns

In [4]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    "Age", "DailyRate", "DistanceFromHome", "HourlyRate", "MonthlyIncome",
    "MonthlyRate", "NumCompaniesWorked", "PercentSalaryHike", "TotalWorkingYears",
    "TrainingTimesLastYear", "YearsAtCompany", "YearsInCurrentRole",
    "YearsSinceLastPromotion", "YearsWithCurrManager",
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(X_train_scaled[num_cols].mean().round(2).head(5))
print(X_train_scaled[num_cols].std().round(2).head(5))

Age                -0.0
DailyRate           0.0
DistanceFromHome   -0.0
HourlyRate         -0.0
MonthlyIncome      -0.0
dtype: float64
Age                 1.0
DailyRate           1.0
DistanceFromHome    1.0
HourlyRate          1.0
MonthlyIncome       1.0
dtype: float64


Handle the class imbalance(SMOTE)

In [5]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print()
print("After SMOTE:")
print(y_train_sm.value_counts())
print("New training shape:", X_train_sm.shape)

Before SMOTE:
Attrition
0    986
1    190
Name: count, dtype: int64

After SMOTE:
Attrition
0    986
1    986
Name: count, dtype: int64
New training shape: (1972, 53)


In [6]:
df_enc["IncomeToExperience"] = df_enc["MonthlyIncome"] / (df_enc["TotalWorkingYears"] + 1)

df_enc["PromotionDelay"] = df_enc["YearsSinceLastPromotion"] / (df_enc["YearsAtCompany"] + 1)

df_enc["EngagementScore"] = df_enc[
    ["JobSatisfaction", "EnvironmentSatisfaction", "RelationshipSatisfaction", "JobInvolvement"]
].mean(axis=1)

df_enc["WorkloadStress"] = (
    (df_enc["OverTime"] == 1) & (df_enc["BusinessTravel_Travel_Frequently"] == 1)
).astype(int)

print(df_enc.shape)
print(df_enc[["IncomeToExperience", "PromotionDelay", "EngagementScore", "WorkloadStress"]].head())

(1470, 54)
   IncomeToExperience  PromotionDelay  EngagementScore  WorkloadStress
0          665.888889        0.000000             2.50               0
1          466.363636        0.090909             2.75               0
2          261.250000        0.000000             2.75               0
3          323.222222        0.333333             3.25               1
4          495.428571        0.666667             2.50               0


In [7]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    "Age", "DailyRate", "DistanceFromHome", "HourlyRate", "MonthlyIncome",
    "MonthlyRate", "NumCompaniesWorked", "PercentSalaryHike", "TotalWorkingYears",
    "TrainingTimesLastYear", "YearsAtCompany", "YearsInCurrentRole",
    "YearsSinceLastPromotion", "YearsWithCurrManager",
    "IncomeToExperience", "PromotionDelay", "EngagementScore",
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(X_train_scaled[num_cols].mean().round(2).tail(3))
print(X_train_scaled[num_cols].std().round(2).tail(3))

IncomeToExperience    0.0
PromotionDelay        0.0
EngagementScore      -0.0
dtype: float64
IncomeToExperience    1.0
PromotionDelay        1.0
EngagementScore       1.0
dtype: float64


In [10]:
import os
import joblib

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

X_train_sm.to_csv("../data/processed/X_train_smote.csv", index=False)
y_train_sm.to_csv("../data/processed/y_train_smote.csv", index=False)
X_train_scaled.to_csv("../data/processed/X_train_scaled.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test_scaled.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(list(X_train.columns), "../models/feature_columns.pkl")

print("Saved. X_train_smote shape:", X_train_sm.shape)

Saved. X_train_smote shape: (1972, 53)


In [9]:
print("X_train:", X_train.shape)
print("X_train_scaled:", X_train_scaled.shape)
print("X_train_sm:", X_train_sm.shape)
print("num_cols count:", len(num_cols))

X_train: (1176, 53)
X_train_scaled: (1176, 53)
X_train_sm: (1972, 53)
num_cols count: 17
